In [ ]:
from __future__ import division
import json
%matplotlib inline
import numpy as np
import pandas as pd
from collections import defaultdict, Counter
import itertools, operator
import csv
import os,glob
import datetime
import geopandas as gpd
from shapely import wkt
from scipy.stats import linregress, spearmanr
import matplotlib.pyplot as plt
import seaborn as sns

import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import dask.dataframe as dd



import networkx as nx
import collections

from scipy.stats import gaussian_kde

from scipy.spatial import cKDTree

import os
import sys

# INFORMATION

In [ ]:
date = '_08_2023'

# OPEN MOBILITY DATA

In [ ]:
#only OUR DISTRICTS

os.chdir(r'C:\Users\mduran\Desktop\PhD\PROJECT 2\DATA\districts')

results_districts = pd.read_csv('results_districts'+date+'.csv')

esp_ids = set(results_districts.ID.unique())

In [ ]:
import os, re, pandas as pd

os.chdir(r'C:\Users\mduran\Desktop\PhD\PROJECT 2\DATA\mobility')
path   = '202308_Viajes_distritos'
fields = ['origen','destino','actividad_origen','actividad_destino','renta','edad','viajes']

dfs = []
for fn in sorted(os.listdir(path)):
    if not re.match(r'^202308\d{2}_Viajes_distritos', fn):
        continue

    day = int(fn[6:8])   
    print(day)
    if not (1 <= day <= 31):  
        continue

    full = os.path.join(path, fn)
    dff  = pd.read_csv(full, delimiter='|', usecols=fields)
    dff = dff[(dff.origen.isin(esp_ids)) & (dff.destino.isin(esp_ids))]
    dff['day'] = day
    dfs.append(dff)

mob = pd.concat(dfs, ignore_index=True)

In [ ]:
dfs = []
dff = []

In [ ]:
mob

In [ ]:
print("ANTES")
mob.info(memory_usage='deep')

obj_cols = mob.select_dtypes(include=['object']).columns.tolist()
print("Columnas object:", obj_cols)

for col in obj_cols:
    print(col, "→", mob[col].nunique(), "valores únicos")

to_cat = ['origen','destino','actividad_origen','actividad_destino','renta','edad']
for col in to_cat:
    mob[col] = mob[col].astype('category')
 
print("\nDESPUÉS")
mob.info(memory_usage='deep')

In [ ]:
to_cat = ['origen','destino','actividad_origen','actividad_destino','renta','edad']
for col in to_cat:
    mob[col] = mob[col].astype('category')

print("\nDESPUÉS")
mob.info(memory_usage='deep')

In [ ]:
# drop income or age NaN

mob = mob[(~pd.isna(mob.edad))&(~pd.isna(mob.renta))]

In [ ]:
mob = mob.groupby(['origen','destino','actividad_origen','actividad_destino','renta','edad','day']).viajes.sum().to_frame().reset_index() 

In [ ]:
to_obj = ['origen','destino',
          'actividad_origen','actividad_destino',
          'renta','edad']

for col in to_obj:
    mob[col] = mob[col].astype('object')

## ADD DISTANCE AND LOCATION TO EACH DISTRICT

In [ ]:
os.chdir(r'C:\Users\mduran\Desktop\PhD\PROJECT 2\DATA\zonification')

cont = gpd.read_file('zonificacion_distritos.shp',encoding='utf-8',
                   dtype={'ID': str}, separator='|')

cont = cont[cont.ID.isin(esp_ids)]  

cont['geometry'] = cont['geometry'].centroid

cont['x'] = cont['geometry'].x
cont['y'] = cont['geometry'].y

In [ ]:
mob['origen'] = mob['origen'].astype(str)
mob['destino'] = mob['destino'].astype(str)
cont['ID'] = cont['ID'].astype(str)

mob = pd.merge(mob, cont[['ID', 'x', 'y']], left_on='origen', right_on='ID', how='left')
mob.rename(columns={'x': 'x_orig', 'y': 'y_orig'}, inplace=True)

mob = pd.merge(mob, cont[['ID', 'x', 'y']], left_on='destino', right_on='ID', how='left', suffixes=('_orig', '_dest'))
mob.rename(columns={'x': 'x_dest', 'y': 'y_dest'}, inplace=True)

mob['distance'] = np.sqrt((mob['x_dest'] - mob['x_orig'])**2 + (mob['y_dest'] - mob['y_orig'])**2)

mob.drop(columns=['ID_orig', 'ID_dest'], inplace=True)

In [ ]:
mob

# SAVE DATA

In [ ]:
# SAVE MOB DATAFRAMES

os.chdir(r'C:\Users\mduran\Desktop\PhD\PROJECT 2')

mob.to_csv('mob_daily'+date+'.csv', index=False)

In [ ]:
mob